In [ ]:
# Use in Databricks serverless
#%pip install tqdm

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
import glob
from tqdm import tqdm

# Assumes you have a SparkSession named 'spark' available
spark = SparkSession.builder.getOrCreate()

In [ ]:
catalog_name = "use1_prod_artemis_catalog_3718194974443840"
project_dir = f"/Volumes/{catalog_name}/production/data/"
table_name = f"{catalog_name}.production.annotations_table"

In [ ]:
img_list = glob.glob(f"{project_dir}*/annotations/*/*/*/images/*.jpg")

In [ ]:
len(img_list)

In [ ]:
img_df = pd.DataFrame({
    'image_path': img_list
})

img_df.head()

In [ ]:
for row in tqdm(img_df.itertuples(), total=len(img_df)):
    idx = row[0]
    split_path = row[1].split('/')
    b_name = split_path[-1]
    img_df.loc[idx, 'annotation_id'] = b_name.split('.')[0]
    img_df.loc[idx, 'project'] = split_path[5]
    img_df.loc[idx, 'annotation_task'] = split_path[8]
    img_df.loc[idx, 'annotation_batch'] = split_path[9]
    img_df.loc[idx, 'annotation_type'] = split_path[7]

img_df.head()

In [ ]:
anno_list = glob.glob(f"{project_dir}*/annotations/*/*/*/*/*.json")
len(anno_list)

In [ ]:
anno_df = pd.DataFrame({
    'annotation_path': anno_list
})

anno_df.head()

In [ ]:
for row in tqdm(anno_df.itertuples(), total=len(anno_df)):
    idx = row[0]
    split_path = row[1].split('/')
    b_name = split_path[-1]
    anno_df.loc[idx, 'annotation_id'] = b_name.split('.')[0]
    anno_df.loc[idx, 'download_date'] = split_path[-2]

In [ ]:
anno_df = anno_df[~anno_df['download_date'].isin(['images'])]
anno_df['download_date'] = anno_df['download_date'].apply(lambda x: x.replace('_', '-'))
anno_df.loc[:, 'download_date'] = pd.to_datetime(anno_df['download_date'])
anno_df.head()

In [ ]:
anno_df = anno_df.sort_values(['annotation_id', 'download_date'], ascending=False)
anno_df.head()

In [ ]:
dedup_anno_df = anno_df.drop_duplicates(subset=['annotation_id'], keep='first')
dedup_anno_df.head()

In [ ]:
final_df = img_df.merge(dedup_anno_df, on='annotation_id', how='left')


In [ ]:
assert len(img_df) == len(final_df)

In [ ]:
# Convert the pandas DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(final_df)

# You can now see the schema of the new Spark DataFrame
spark_df.printSchema()

In [ ]:
spark_df.write.saveAsTable(table_name,
                           mode="overwrite")